# Clasificación de Fashion-MNIST

Red densa con validación, detención temprana y análisis por clase.

## Dependencias

En un entorno nuevo: `%pip install tensorflow numpy matplotlib scikit-learn`. La primera carga de Fashion-MNIST descarga el dataset.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train_all = x_train_all.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train, x_val = x_train_all[:-6000], x_train_all[-6000:]
y_train, y_val = y_train_all[:-6000], y_train_all[-6000:]
print(x_train.shape, x_val.shape, x_test.shape)

La prueba queda aislada. En un proyecto real se debe justificar que las particiones representan el escenario de despliegue.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(10),
])
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30, batch_size=128, callbacks=[early_stop], verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validación')
axes[0].set(title='Pérdida', xlabel='Época')
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='validación')
axes[1].set(title='Accuracy', xlabel='Época')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Evaluación final

La prueba se usa después de fijar arquitectura, optimizador y punto de detención.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
logits = model.predict(x_test, verbose=0)
y_pred = np.argmax(logits, axis=1)
print(f'loss={test_loss:.4f}, accuracy={test_accuracy:.4f}')
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, normalize='true', values_format='.2f', cmap='Blues', ax=ax
)
plt.show()

## Trabajo propuesto

Compare esta red con regresión logística y una CNN pequeña. Mantenga la misma partición y reporte accuracy, parámetros, tiempo y errores por clase.